In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

DB = "workspace.complaints"
SILVER = f"{DB}.silver_complaints"
LOGS = f"{DB}.pipeline_execution_logs"

DIM_PRODUCT = f"{DB}.dim_product"
DIM_COMPANY = f"{DB}.dim_company"
DIM_LOCATION = f"{DB}.dim_location"
DIM_DATE = f"{DB}.dim_date"
FACT_COMPLAINTS = f"{DB}.fact_complaints"

BATCH_ID = f"batch_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)

print("Starting Silver to Gold Transformation...")

try:
    df_silver = spark.table(SILVER)
    rows_read = df_silver.count()

    # 1. Build dim_product (Strictly Product & Sub-product)
    df_product = (
        df_silver.select("product", "sub_product")
        .distinct()
        .withColumn("product_id", F.sha2(F.concat_ws("||", 
            F.coalesce(F.col("product"), F.lit("__NULL__")), 
            F.coalesce(F.col("sub_product"), F.lit("__NULL__"))), 256))
    )
    df_product.write.format("delta").mode("overwrite").saveAsTable(DIM_PRODUCT)

    # 2. Build dim_company (Company name only; responses moved to fact)
    df_company = (
        df_silver.select("company")
        .distinct()
        .withColumn("company_id", F.sha2(F.coalesce(F.col("company"), F.lit("__NULL__")), 256))
    )
    df_company.write.format("delta").mode("overwrite").saveAsTable(DIM_COMPANY)

    # 3. Build dim_location
    df_location = (
        df_silver.select("state", "zip_code")
        .distinct()
        .withColumn("location_id", F.sha2(F.concat_ws("||", 
            F.coalesce(F.col("state"), F.lit("__NULL__")), 
            F.coalesce(F.col("zip_code"), F.lit("__NULL__"))), 256))
    )
    df_location.write.format("delta").mode("overwrite").saveAsTable(DIM_LOCATION)

    # 4. Build dim_date from distinct dates
    df_dates = (
        df_silver.select(F.to_date("date_received").alias("dt"))
        .union(df_silver.select(F.to_date("date_sent_to_company").alias("dt")))
        .filter(F.col("dt").isNotNull())
        .distinct()
    )
    df_date = (
        df_dates
        .withColumn("date_id", F.date_format("dt", "yyyyMMdd").cast("long"))
        .withColumn("date", F.col("dt"))
        .withColumn("year", F.year("dt"))
        .withColumn("quarter", F.quarter("dt"))
        .withColumn("month", F.month("dt"))
        .withColumn("month_name", F.date_format("dt", "MMMM"))
        .withColumn("day", F.dayofmonth("dt"))
        .withColumn("day_of_week", F.date_format("dt", "EEEE"))
        .drop("dt")
    )
    df_date.write.format("delta").mode("overwrite").saveAsTable(DIM_DATE)

    # 5. Build fact_complaints with Surrogate Keys, Issue Attributes, and Responses
    df_fact = (
        df_silver.alias("s")
        .join(df_product.alias("p"), 
              (F.coalesce(F.col("s.product"), F.lit("__NULL__")) == F.coalesce(F.col("p.product"), F.lit("__NULL__"))) &
              (F.coalesce(F.col("s.sub_product"), F.lit("__NULL__")) == F.coalesce(F.col("p.sub_product"), F.lit("__NULL__"))), "inner")
        .join(df_company.alias("c"),
              F.coalesce(F.col("s.company"), F.lit("__NULL__")) == F.coalesce(F.col("c.company"), F.lit("__NULL__")), "inner")
        .join(df_location.alias("l"),
              (F.coalesce(F.col("s.state"), F.lit("__NULL__")) == F.coalesce(F.col("l.state"), F.lit("__NULL__"))) &
              (F.coalesce(F.col("s.zip_code"), F.lit("__NULL__")) == F.coalesce(F.col("l.zip_code"), F.lit("__NULL__"))), "inner")
        .select(
            F.col("s.complaint_id"),
            F.col("p.product_id"),
            F.col("c.company_id"),
            F.col("l.location_id"),
            F.date_format("s.date_received", "yyyyMMdd").cast("long").alias("date_received_id"),
            F.date_format("s.date_sent_to_company", "yyyyMMdd").cast("long").alias("date_sent_id"),
            # Descriptive attributes & complaint outcomes stored in Fact
            F.col("s.issue"),
            F.col("s.sub_issue"),
            F.col("s.company_response"),
            F.col("s.company_public_response"),
            F.col("s.submitted_via"),
            F.when(F.col("s.timely") == True, 1).otherwise(0).alias("timely_flag"),
            F.datediff(F.col("s.date_sent_to_company"), F.col("s.date_received")).alias("days_to_send"),
            F.current_timestamp().alias("load_timestamp")
        )
    )

    # Idempotent MERGE into Fact Table and accurate metrics extraction
    if not spark.catalog.tableExists(FACT_COMPLAINTS):
        df_fact.write.format("delta").mode("overwrite").saveAsTable(FACT_COMPLAINTS)
        rows_inserted = df_fact.count()
        rows_updated = 0
    else:
        fact_table = DeltaTable.forName(spark, FACT_COMPLAINTS)
        fact_table.alias("t").merge(
            df_fact.alias("s"),
            "t.complaint_id = s.complaint_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        
        # Extract precise metrics from Delta history
        metrics = spark.sql(f"DESCRIBE HISTORY {FACT_COMPLAINTS} LIMIT 1").select("operationMetrics").first()[0]
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated = int(metrics.get("numTargetRowsUpdated", 0))

    # Optional Performance Optimization (OPTIMIZE & Z-ORDER)
    print("Optimizing Gold Fact Table...")
    spark.sql(f"OPTIMIZE {FACT_COMPLAINTS} ZORDER BY (date_received_id, product_id)")

    status = "Success"
    error_msg = None
    print(f"Gold Star Schema successfully updated. Inserted: {rows_inserted}, Updated: {rows_updated}.")

except Exception as e:
    status = "Failure"
    error_msg = str(e)[:2000]
    rows_read = rows_inserted = rows_updated = 0
    print(f"Gold layer build failed: {error_msg}")
    raise e

finally:
    END_TIME = datetime.now(timezone.utc)
    duration_seconds = round((END_TIME - START_TIME).total_seconds(), 2)
    
    spark.createDataFrame([(
        RUN_ID, BATCH_ID, "gold", "transform", "star_schema",
        START_TIME, END_TIME, float(duration_seconds), status,
        int(rows_read), int(rows_inserted), int(rows_updated), 0, error_msg
    )], schema="""run_id string, batch_id string, layer string, load_type string, parameter string, 
                  start_time timestamp, end_time timestamp, duration_seconds double, status string, 
                  rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, error_message string""") \
    .write.format("delta").mode("append").saveAsTable(LOGS)

dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
-- SELECT * FROM workspace.complaints.dim_product  LIMIT 20;
SELECT * FROM workspace.complaints.dim_company  LIMIT 20;
-- SELECT * FROM workspace.complaints.dim_location LIMIT 20;
-- SELECT * FROM workspace.complaints.dim_date ORDER BY `date` DESC LIMIT 20;

In [0]:
%sql
SELECT 'dim_product' AS tbl, COUNT(*) AS n FROM workspace.complaints.dim_product
UNION ALL SELECT 'dim_company',  COUNT(*) FROM workspace.complaints.dim_company
UNION ALL SELECT 'dim_location', COUNT(*) FROM workspace.complaints.dim_location
UNION ALL SELECT 'dim_date',     COUNT(*) FROM workspace.complaints.dim_date
UNION ALL SELECT 'fact_complaints', COUNT(*) FROM workspace.complaints.fact_complaints
UNION ALL SELECT 'silver_complaints', COUNT(*) FROM workspace.complaints.silver_complaints;

In [0]:
%sql
SELECT f.complaint_id, d.`date` AS received_date, p.product, p.issue,
       c.company, l.state, f.timely_flag, f.days_to_send
FROM workspace.complaints.fact_complaints f
JOIN workspace.complaints.dim_product  p ON f.product_id = p.product_id
JOIN workspace.complaints.dim_company  c ON f.company_id = c.company_id
JOIN workspace.complaints.dim_location l ON f.location_id = l.location_id
JOIN workspace.complaints.dim_date     d ON f.date_received_id = d.date_id
LIMIT 20;